# 3. Cluster Interpretation and Graph Artifact Construction

군집별 키워드·대표 메시지를 해석하고, GNN 학습에 사용할 node/edge 테이블을 생성합니다.

## 입력 파일

- `data/processed/message_master_minimal_final_clustered.csv`
- `data/raw/*.xlsx`

## 출력 파일

- `results/cluster_interpretation/cluster_profile_for_interpretation.xlsx`
- `results/cluster_interpretation/gnn_nodes.csv`
- `results/cluster_interpretation/gnn_edges.csv`
- `results/cluster_interpretation/message_with_cluster_name.csv`


## 1. Imports

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Path configuration

In [ ]:
# =========================
# Project path configuration
# =========================

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR = PROJECT_ROOT / "results" / "cluster_interpretation"

OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "HEAD": RAW_DATA_DIR / "헤드메세지 MASTER 원본.xlsx",
    "DR": RAW_DATA_DIR / "DR_Data.xlsx",
    "RULE": RAW_DATA_DIR / "Final_Rule_Deduction_Clean_add.xlsx",
    "ITEM": RAW_DATA_DIR / "ITHM MASTER.xlsx",
}

CLUSTERED_MESSAGE_PATH = PROCESSED_DATA_DIR / "message_master_minimal_final_clustered.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_DIR:", OUT_DIR)

## 3. Load data

In [ ]:
# =========================
# Load stage 2 outputs and raw source files
# =========================

message_df = pd.read_csv(CLUSTERED_MESSAGE_PATH)
head_ms = pd.read_excel(RAW_FILES["HEAD"], engine="openpyxl")
DR = pd.read_excel(RAW_FILES["DR"], engine="openpyxl", header=1)
Rule = pd.read_excel(RAW_FILES["RULE"], engine="openpyxl", header=2)
item = pd.read_excel(RAW_FILES["ITEM"], engine="openpyxl")

In [ ]:
# Output directory is configured above.
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("저장 폴더:", OUT_DIR)

## 4. Text normalization and cluster field validation

In [ ]:
print("message_df columns")
print(message_df.columns.tolist())

print("\nhead_ms columns")
print(head_ms.columns.tolist())

print("\nDR columns")
print(DR.columns.tolist())

print("\nRule columns")
print(Rule.columns.tolist())

print("\nitem columns")
print(item.columns.tolist())

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def norm_key(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", "", x)
    x = x.replace("\n", "")
    return x

def row_get(row, *candidates):
    lookup = {norm_key(c): c for c in row.index}

    for cand in candidates:
        key = norm_key(cand)
        if key in lookup:
            return row[lookup[key]]

    return np.nan

def value_counts_text(series, top_n=10):
    vc = series.dropna().astype(str).value_counts().head(top_n)
    return " | ".join([f"{idx}({cnt})" for idx, cnt in vc.items()])

## 5. Cluster distribution and keyword extraction

In [ ]:
# message_id 없으면 생성
if "message_id" not in message_df.columns:
    message_df["message_id"] = np.arange(len(message_df))

# final_cluster 없으면 kmeans_cluster_10 사용
if "final_cluster" not in message_df.columns:
    if "kmeans_cluster_10" in message_df.columns:
        message_df["final_cluster"] = message_df["kmeans_cluster_10"]
    else:
        raise ValueError("final_cluster 또는 kmeans_cluster_10 컬럼이 없습니다.")

message_df["final_cluster"] = message_df["final_cluster"].astype(int)

# message_text 없으면 text_to_embed 사용
if "message_text" not in message_df.columns:
    if "text_to_embed" in message_df.columns:
        message_df["message_text"] = message_df["text_to_embed"]
    else:
        raise ValueError("message_text 또는 text_to_embed 컬럼이 없습니다.")

message_df["message_text"] = message_df["message_text"].astype(str).map(clean_text)

# 선택 컬럼 보정
if "source_domain" not in message_df.columns:
    message_df["source_domain"] = "UNKNOWN"

if "message_type" not in message_df.columns:
    message_df["message_type"] = "UNKNOWN"

if "date" not in message_df.columns:
    message_df["date"] = np.nan

if "row_idx_in_source" not in message_df.columns:
    message_df["row_idx_in_source"] = np.nan

message_df[["message_id", "source_domain", "row_idx_in_source", "date", "message_type", "final_cluster", "message_text"]].head()

In [ ]:
cluster_dist = (
    message_df["final_cluster"]
    .value_counts()
    .sort_index()
    .reset_index()
)

cluster_dist.columns = ["final_cluster", "count"]
cluster_dist["ratio"] = cluster_dist["count"] / cluster_dist["count"].sum()

display(cluster_dist)

In [ ]:
def get_cluster_keywords(
    df,
    cluster_col="final_cluster",
    text_col="message_text",
    top_n=20,
    max_features=3000
):
    cluster_texts = (
        df.groupby(cluster_col)[text_col]
        .apply(lambda x: " ".join(x.astype(str).map(clean_text)))
        .reset_index()
    )

    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b[\w가-힣A-Za-z0-9\-\+\/\.%]+\b"
    )

    tfidf = vectorizer.fit_transform(cluster_texts[text_col])
    terms = vectorizer.get_feature_names_out()

    rows = []

    for i, cid in enumerate(cluster_texts[cluster_col]):
        row = tfidf[i].toarray().ravel()
        top_idx = row.argsort()[-top_n:][::-1]
        keywords = [terms[j] for j in top_idx if row[j] > 0]

        rows.append({
            cluster_col: cid,
            "top_keywords": ", ".join(keywords)
        })

    return pd.DataFrame(rows)

In [ ]:
cluster_keywords = get_cluster_keywords(
    message_df,
    cluster_col="final_cluster",
    text_col="message_text",
    top_n=20
)

display(cluster_keywords)

## 6. Cluster interpretation

In [ ]:
def get_cluster_examples(
    df,
    cluster_col="final_cluster",
    text_col="message_text",
    n_examples=15
):
    rows = []

    for cid in sorted(df[cluster_col].dropna().unique()):
        temp = df[df[cluster_col] == cid].copy()

        examples = (
            temp[text_col]
            .dropna()
            .astype(str)
            .head(n_examples)
            .tolist()
        )

        rows.append({
            cluster_col: cid,
            "count": len(temp),
            "examples": " || ".join(examples)
        })

    return pd.DataFrame(rows)

In [ ]:
cluster_examples = get_cluster_examples(
    message_df,
    cluster_col="final_cluster",
    text_col="message_text",
    n_examples=15
)

display(cluster_examples)

In [ ]:
cluster_profile = (
    message_df
    .groupby("final_cluster")
    .agg(
        count=("final_cluster", "size"),
        source_domain_dist=("source_domain", lambda x: value_counts_text(x, top_n=10)),
        message_type_dist=("message_type", lambda x: value_counts_text(x, top_n=10)),
        date_dist=("date", lambda x: value_counts_text(x, top_n=5)),
        sample_message=("message_text", lambda x: " || ".join(x.astype(str).head(5)))
    )
    .reset_index()
    .sort_values("final_cluster")
)

cluster_profile = cluster_profile.merge(
    cluster_keywords,
    on="final_cluster",
    how="left"
)

cluster_profile = cluster_profile.merge(
    cluster_examples[["final_cluster", "examples"]],
    on="final_cluster",
    how="left"
)

display(cluster_profile)

In [ ]:
# ==============================
# LLM 기반 의미 해석 설정
# ==============================
# 기존 방식:
# - top_keywords에 특정 단어가 포함되면 군집명을 반환하는 규칙 기반 방식
#
# 변경 방식:
# - 군집별 대표 키워드, 대표 메시지, message_type/source 분포를 LLM에 전달
# - LLM이 의미를 종합하여 군집명, 문제유형, 해석 메모를 JSON으로 반환
#
# 실행 전 확인:
# 1) Ollama 서버 실행: ollama serve
# 2) 모델 설치/확인: ollama list
# 3) 아래 GEMMA_MODEL 이름이 실제 설치된 모델명과 일치해야 함

import json
import time
import hashlib
from pathlib import Path

import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
GEMMA_MODEL = "gemma4:e2b"  # 예: gemma:2b, gemma2:9b, gemma3:4b

LLM_TIMEOUT = 120
LLM_TEMPERATURE = 0.1
LLM_NUM_PREDICT = 700
RAISE_ON_LLM_ERROR = False

LLM_CACHE_PATH = OUT_DIR / "cluster_llm_interpretation_cache.json"

In [ ]:
def truncate_text(text, max_chars=1800):
    """프롬프트가 과도하게 길어지지 않도록 텍스트 길이를 제한합니다."""
    text = clean_text(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ...[TRUNCATED]"

In [ ]:
def load_llm_cache(cache_path=LLM_CACHE_PATH):
    if Path(cache_path).exists():
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

In [ ]:
def save_llm_cache(cache, cache_path=LLM_CACHE_PATH):
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

In [ ]:
def parse_llm_json(raw_text):
    """
    Ollama 응답에서 JSON 객체를 안정적으로 추출합니다.
    모델이 ```json ... ``` 형태나 앞뒤 설명을 붙여도 최대한 복구합니다.
    """
    raw_text = clean_text(raw_text)
    raw_text = raw_text.replace("```json", "").replace("```JSON", "").replace("```", "").strip()

    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        start = raw_text.find("{")
        end = raw_text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(raw_text[start:end + 1])
        raise

In [ ]:
def build_cluster_prompt(row, max_examples=12):
    examples_raw = str(row.get("examples", ""))
    examples = [truncate_text(x, 220) for x in examples_raw.split(" || ") if clean_text(x) != ""]
    examples = examples[:max_examples]

    examples_text = "\n".join([f"- {ex}" for ex in examples])

    prompt = f"""
너는 제조/SCM/재고관리 데이터를 해석하는 데이터 분석가다.
아래는 KMeans로 묶인 하나의 메시지 군집 정보다.

중요한 원칙:
- 단순히 특정 키워드 하나가 포함됐다고 라벨을 붙이지 말 것.
- top_keywords, 대표 메시지, message_type 분포, source_domain 분포를 종합해서 의미 기반으로 판단할 것.
- 군집명은 업무 담당자가 바로 이해할 수 있는 한국어 명사구로 작성할 것.
- 문제유형은 이 군집이 나타내는 관리상 이슈를 한 문장으로 작성할 것.
- 근거 신호는 군집 해석에 실제로 사용한 의미 단서만 적을 것.
- 반드시 JSON 객체 하나만 반환할 것. Markdown, 설명문, 코드블록은 쓰지 말 것.

[군집 정보]
final_cluster: {row.get("final_cluster")}
count: {row.get("count")}
source_domain_dist: {truncate_text(row.get("source_domain_dist", ""), 700)}
message_type_dist: {truncate_text(row.get("message_type_dist", ""), 700)}
date_dist: {truncate_text(row.get("date_dist", ""), 500)}
top_keywords: {truncate_text(row.get("top_keywords", ""), 900)}
sample_message: {truncate_text(row.get("sample_message", ""), 1200)}

[대표 메시지 예시]
{examples_text}

[반환 JSON 스키마]
{{
  "cluster_name": "15~35자 내외의 한국어 군집명",
  "problem_type": "이 군집이 의미하는 관리/운영상 문제유형 한 문장",
  "interpretation_memo": "왜 그렇게 해석했는지 1~2문장",
  "representative_signals": ["의미 근거1", "의미 근거2", "의미 근거3"],
  "confidence": 0.0
}}
""".strip()

    return prompt

In [ ]:
def call_ollama_json(prompt, retries=2):
    payload = {
        "model": GEMMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": LLM_TEMPERATURE,
            "num_predict": LLM_NUM_PREDICT
        }
    }

    last_error = None

    for attempt in range(retries + 1):
        try:
            response = requests.post(
                OLLAMA_URL,
                json=payload,
                timeout=LLM_TIMEOUT
            )
            response.raise_for_status()

            response_json = response.json()
            raw_answer = response_json.get("response", "")
            parsed = parse_llm_json(raw_answer)

            return parsed, raw_answer

        except Exception as e:
            last_error = e
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))

    raise RuntimeError(f"Ollama LLM 호출 또는 JSON 파싱 실패: {last_error}")

In [ ]:
def normalize_llm_result(parsed):
    """LLM JSON 결과를 노트북에서 쓰기 좋은 형태로 정규화합니다."""
    cluster_name = clean_text(parsed.get("cluster_name", ""))
    problem_type = clean_text(parsed.get("problem_type", ""))
    interpretation_memo = clean_text(parsed.get("interpretation_memo", ""))

    representative_signals = parsed.get("representative_signals", [])
    if isinstance(representative_signals, list):
        representative_signals = " | ".join([clean_text(x) for x in representative_signals if clean_text(x) != ""])
    else:
        representative_signals = clean_text(representative_signals)

    try:
        confidence = float(parsed.get("confidence", np.nan))
    except Exception:
        confidence = np.nan

    return {
        "llm_cluster_name": cluster_name,
        "llm_problem_type": problem_type,
        "llm_interpretation_memo": interpretation_memo,
        "llm_representative_signals": representative_signals,
        "llm_confidence": confidence
    }

In [ ]:
def interpret_cluster_with_llm(row, cache):
    cid = int(row["final_cluster"])
    prompt = build_cluster_prompt(row)

    fingerprint = hashlib.md5(prompt.encode("utf-8")).hexdigest()
    cache_key = f"cluster_{cid}_{fingerprint}"

    if cache_key in cache:
        cached = cache[cache_key]
        result = cached["result"].copy()
        result["llm_cache_hit"] = True
        return result

    try:
        parsed, raw_answer = call_ollama_json(prompt)
        result = normalize_llm_result(parsed)
        result["llm_raw_response"] = raw_answer
        result["llm_error"] = ""
        result["llm_cache_hit"] = False

    except Exception as e:
        if RAISE_ON_LLM_ERROR:
            raise

        result = {
            "llm_cluster_name": f"cluster_{cid}_LLM해석확인필요",
            "llm_problem_type": "LLM 호출 실패로 문제유형 확인 필요",
            "llm_interpretation_memo": "Ollama 서버, 모델명, JSON 응답 형식을 확인해야 합니다.",
            "llm_representative_signals": "",
            "llm_confidence": np.nan,
            "llm_raw_response": "",
            "llm_error": str(e),
            "llm_cache_hit": False
        }

    cache[cache_key] = {
        "final_cluster": cid,
        "model": GEMMA_MODEL,
        "result": result
    }
    save_llm_cache(cache)

    return result

In [ ]:
def interpret_all_clusters_with_llm(cluster_profile):
    cache = load_llm_cache()
    rows = []

    for _, row in cluster_profile.sort_values("final_cluster").iterrows():
        cid = int(row["final_cluster"])
        print(f"LLM 의미 해석 중: cluster {cid}")

        result = interpret_cluster_with_llm(row, cache)
        result["final_cluster"] = cid
        rows.append(result)

    return pd.DataFrame(rows)

## 7. Original table part extraction

In [ ]:
# ==============================
# 군집별 LLM 의미 해석 실행
# ==============================
# 이 셀은 군집 수만큼 Ollama를 호출합니다.
# 같은 입력에 대해서는 cache 파일을 사용하므로 재실행 시 중복 호출을 줄입니다.

llm_cluster_interpretation = interpret_all_clusters_with_llm(cluster_profile)

# 재실행 시 중복 컬럼 생성을 방지
drop_cols = [
    "cluster_name_draft",
    "cluster_name_final",
    "problem_type",
    "interpretation_memo",
    "llm_cluster_name",
    "llm_problem_type",
    "llm_interpretation_memo",
    "llm_representative_signals",
    "llm_confidence",
    "llm_raw_response",
    "llm_error",
    "llm_cache_hit"
]

cluster_profile = cluster_profile.drop(columns=[c for c in drop_cols if c in cluster_profile.columns], errors="ignore")

cluster_profile = cluster_profile.merge(
    llm_cluster_interpretation,
    on="final_cluster",
    how="left"
)

cluster_profile["cluster_name_final"] = cluster_profile["llm_cluster_name"].replace("", np.nan)
cluster_profile["cluster_name_final"] = cluster_profile["cluster_name_final"].fillna(
    cluster_profile["final_cluster"].map(lambda x: f"cluster_{x}_LLM확인필요")
)

cluster_profile["problem_type"] = cluster_profile["llm_problem_type"].replace("", np.nan)
cluster_profile["problem_type"] = cluster_profile["problem_type"].fillna("LLM 기반 문제유형 확인 필요")

cluster_profile["interpretation_memo"] = cluster_profile["llm_interpretation_memo"].replace("", np.nan)
cluster_profile["interpretation_memo"] = cluster_profile["interpretation_memo"].fillna(
    "LLM 기반 의미 해석 결과입니다. llm_error가 있으면 Ollama 실행 상태를 확인하세요."
)

display(
    cluster_profile[
        [
            "final_cluster",
            "count",
            "cluster_name_final",
            "problem_type",
            "interpretation_memo",
            "llm_representative_signals",
            "llm_confidence",
            "llm_error",
            "source_domain_dist",
            "message_type_dist",
            "top_keywords",
            "examples"
        ]
    ]
)


In [ ]:
# ==============================
# LLM 해석 결과를 전체 메시지 데이터에 반영
# ==============================
# 기존 수동 cluster_name_map / cluster_problem_map 대신
# LLM이 생성한 cluster_name_final / problem_type을 매핑합니다.

cluster_name_map = (
    cluster_profile
    .set_index("final_cluster")["cluster_name_final"]
    .to_dict()
)

cluster_problem_map = (
    cluster_profile
    .set_index("final_cluster")["problem_type"]
    .to_dict()
)

message_df["cluster_name_final"] = message_df["final_cluster"].map(cluster_name_map)
message_df["problem_type"] = message_df["final_cluster"].map(cluster_problem_map)

display(
    cluster_profile[
        [
            "final_cluster",
            "count",
            "cluster_name_final",
            "problem_type",
            "interpretation_memo",
            "llm_representative_signals",
            "llm_confidence",
            "source_domain_dist",
            "message_type_dist",
            "top_keywords"
        ]
    ]
)

print("LLM 해석 오류 건수:", int(cluster_profile["llm_error"].fillna("").ne("").sum()))


In [ ]:
print("message_df source_domain count")
display(message_df["source_domain"].value_counts())

print("원본 파일 행 수")
print("HEAD:", len(head_ms))
print("DR:", len(DR))
print("RULE:", len(Rule))
print("ITEM:", len(item))

print("\nmessage_df row_idx 범위")
display(
    message_df
    .groupby("source_domain")["row_idx_in_source"]
    .agg(["min", "max", "nunique", "count"])
)

In [ ]:
def extract_head_parts(head_ms):
    rows = []

    for idx, row in head_ms.iterrows():
        base = {
            "source_domain": "HEAD",
            "row_idx_in_source": idx,
            "date_raw": row_get(row, "DATE", "Date"),
            "ORG": row_get(row, "ORG"),
            "Product": row_get(row, "Product"),
            "HeadMessage": row_get(row, "Inventory Trend", "Head Message", "HeadMessage")
        }

        for i in range(1, 6):
            part_no = row_get(row, f"PartNo{i}", f"Part No._{i}", f"Part No.{i}")
            part_desc = row_get(row, f"Description{i}", f"Description.{i}", f"Desc._{i}")

            if pd.notna(part_no) or pd.notna(part_desc):
                rows.append({
                    **base,
                    "part_rank": i,
                    "part_no": part_no,
                    "part_desc": part_desc,
                    "uit": row_get(row, f"UIT{i}", f"UIT_{i}"),
                    "inv_score": row_get(row, f"Inv. Score{i}", f"Inv Score_{i}"),
                    "target_dio": row_get(row, f"Target\nDIO{i}", f"Target DIO_{i}", f"Target DIO{i}"),
                    "estimated_dio": row_get(row, f"Estimated DIO{i}", f"Est DIO_{i}"),
                    "dio_signal": row_get(row, f"DIO Signal{i}", f"Signal_{i}"),
                    "onhand_qty": row_get(row, f"Onhand QTY{i}"),
                    "po_qty": row_get(row, f"PO (M0)\nQTY{i}", f"PO (M0) QTY{i}"),
                    "gross_req_qty": row_get(row, f"Gross Req (M0)\nQTY{i}", f"Gross Req (M0) QTY{i}"),
                    "estimated_inv_qty": row_get(row, f"Estimated Inv. (M0)\nQTY{i}", f"Estimated Inv. (M0) QTY{i}"),
                    "estimated_inv_amt": row_get(row, f"Estimated Inv. (M0)\nAMT{i}", f"Estimated Inv. (M0) AMT{i}")
                })

    return pd.DataFrame(rows)

head_part_long = extract_head_parts(head_ms)

print("head_part_long:", head_part_long.shape)
display(head_part_long.head())

In [ ]:
def extract_item_parts(item):
    rows = []

    for idx, row in item.iterrows():
        base = {
            "source_domain": "ITEM",
            "row_idx_in_source": idx,
            "date_raw": row_get(row, "Date", "DATE"),
            "ORG": row_get(row, "ORG"),
            "Product": row_get(row, "Product"),
            "HeadMessage": row_get(row, "Inventory Trend", "Head Message", "HeadMessage")
        }

        for i in range(1, 6):
            part_no = row_get(row, f"Part No._{i}", f"PartNo{i}", f"Part No.{i}")
            part_desc = row_get(row, f"Desc._{i}", f"Description{i}", f"Description.{i}")

            if pd.notna(part_no) or pd.notna(part_desc):
                rows.append({
                    **base,
                    "part_rank": i,
                    "part_no": part_no,
                    "part_desc": part_desc,
                    "uit": row_get(row, f"UIT_{i}", f"UIT{i}"),
                    "inv_score": row_get(row, f"Inv Score_{i}", f"Inv. Score{i}"),
                    "target_dio": row_get(row, f"Target DIO_{i}", f"Target DIO{i}"),
                    "estimated_dio": row_get(row, f"Est DIO_{i}", f"Estimated DIO{i}"),
                    "dio_signal": row_get(row, f"Signal_{i}", f"DIO Signal{i}"),
                    "estimated_inv_qty": row_get(row, f"Est. Inv Qty_{i}"),
                    "estimated_inv_amt": row_get(row, f"Est. Inv Amt_{i}")
                })

    return pd.DataFrame(rows)

item_part_long = extract_item_parts(item)

print("item_part_long:", item_part_long.shape)
display(item_part_long.head())

In [ ]:
def extract_dr_parts(DR):
    rows = []

    for idx, row in DR.iterrows():
        base = {
            "source_domain": "DR",
            "row_idx_in_source": idx,
            "date_raw": row_get(row, "Date", "DATE"),
            "ORG": row_get(row, "ORG"),
            "Product": row_get(row, "Product"),
            "HeadMessage": row_get(row, "Head Message", "HeadMessage"),
            "category": row_get(row, "Category")
        }

        for i in range(1, 6):
            part_desc = row_get(row, f"Description.{i}", f"Description_{i}", f"Description{i}")

            if pd.notna(part_desc):
                rows.append({
                    **base,
                    "part_rank": i,
                    "part_no": np.nan,
                    "part_desc": part_desc,
                    "po_lt": row_get(row, f"PO L/T.{i}", f"PO LT.{i}"),
                    "po_sum": row_get(row, f"PO SUM.{i}"),
                    "gross_req_sum": row_get(row, f"Gross REQ SUM.{i}"),
                    "po_os_qty": row_get(row, f"PO O&S QTY.{i}"),
                    "po_os_amt": row_get(row, f"PO O&S AMT.{i}")
                })

    return pd.DataFrame(rows)

dr_part_long = extract_dr_parts(DR)

print("dr_part_long:", dr_part_long.shape)
display(dr_part_long.head())

In [ ]:
def extract_rule_parts(Rule):
    rows = []

    for idx, row in Rule.iterrows():
        base = {
            "source_domain": "RULE",
            "row_idx_in_source": idx,
            "date_raw": row_get(row, "Date", "DATE"),
            "ORG": row_get(row, "ORG"),
            "Product": row_get(row, "Product"),
            "HeadMessage": row_get(row, "HeadMessage", "Head Message"),
            "problem": row_get(row, "problem", "Problem")
        }

        for i in range(1, 6):
            part_desc = row_get(row, f"Over Desc_{i}", f"Over Desc.{i}", f"Over Desc{i}")

            if pd.notna(part_desc):
                rows.append({
                    **base,
                    "part_rank": i,
                    "part_no": np.nan,
                    "part_desc": part_desc,
                    "over_amount": row_get(row, f"Over Amount_{i}", f"Over Amount.{i}"),
                    "req_amount": row_get(row, f"REQ Amount_{i}", f"REQ Amount.{i}"),
                    "asset_type": row_get(row, f"Asset_{i}(0:Asset, 1:Non-Asset", f"Asset_{i}")
                })

    return pd.DataFrame(rows)

rule_part_long = extract_rule_parts(Rule)

print("rule_part_long:", rule_part_long.shape)
display(rule_part_long.head())

In [ ]:
part_long = pd.concat(
    [
        head_part_long,
        item_part_long,
        dr_part_long,
        rule_part_long
    ],
    ignore_index=True
)

print("part_long:", part_long.shape)
display(part_long.head())

print("part_long source_domain count")
display(part_long["source_domain"].value_counts())

## 8. Graph node and edge construction

In [ ]:
message_key_cols = [
    "message_id",
    "source_domain",
    "row_idx_in_source",
    "date",
    "message_type",
    "message_text",
    "final_cluster",
    "cluster_name_final",
    "problem_type"
]

message_key = message_df[message_key_cols].copy()

message_key["source_domain"] = message_key["source_domain"].astype(str)
part_long["source_domain"] = part_long["source_domain"].astype(str)

message_part_map = message_key.merge(
    part_long,
    on=["source_domain", "row_idx_in_source"],
    how="left",
    suffixes=("", "_origin")
)

print("message_part_map:", message_part_map.shape)

display(message_part_map.head())

print("매칭 여부")
print("part_desc notna ratio:", round(message_part_map["part_desc"].notna().mean(), 4))

In [ ]:
def make_node_id(node_type, value):
    value = clean_text(value)
    value = value.replace(" ", "_")
    value = re.sub(r"[^\w가-힣\-_\.]", "", value)

    if value == "":
        value = "UNKNOWN"

    return f"{node_type}:{value}"

In [ ]:
nodes = []

# 1. Message node
for _, row in message_df.iterrows():
    nodes.append({
        "node_id": make_node_id("message", row["message_id"]),
        "node_type": "message",
        "node_name": row["message_id"],
        "source_value": row["message_text"]
    })

# 2. Cluster node
for cid in sorted(message_df["final_cluster"].dropna().unique()):
    cname = cluster_name_map.get(cid, f"cluster_{cid}")

    nodes.append({
        "node_id": make_node_id("cluster", cid),
        "node_type": "cluster",
        "node_name": f"cluster_{cid}",
        "source_value": cname
    })

# 3. Source domain node
for v in sorted(message_df["source_domain"].dropna().astype(str).unique()):
    nodes.append({
        "node_id": make_node_id("domain", v),
        "node_type": "source_domain",
        "node_name": v,
        "source_value": v
    })

# 4. Message type node
for v in sorted(message_df["message_type"].dropna().astype(str).unique()):
    nodes.append({
        "node_id": make_node_id("message_type", v),
        "node_type": "message_type",
        "node_name": v,
        "source_value": v
    })

# 5. Date node
for v in sorted(message_df["date"].dropna().astype(str).unique()):
    nodes.append({
        "node_id": make_node_id("date", v),
        "node_type": "date",
        "node_name": v,
        "source_value": v
    })

# 6. ORG / Product / Part node
for col, node_type in [
    ("ORG", "org"),
    ("Product", "product"),
    ("part_no", "part_no"),
    ("part_desc", "part_desc")
]:
    if col in message_part_map.columns:
        for v in sorted(message_part_map[col].dropna().astype(str).unique()):
            if clean_text(v) == "":
                continue

            nodes.append({
                "node_id": make_node_id(node_type, v),
                "node_type": node_type,
                "node_name": v,
                "source_value": v
            })

nodes_df = (
    pd.DataFrame(nodes)
    .drop_duplicates(subset=["node_id"])
    .reset_index(drop=True)
)

print("nodes_df:", nodes_df.shape)
display(nodes_df.head())

print("node_type count")
display(nodes_df["node_type"].value_counts())

In [ ]:
edges = []

def add_edge(source, target, edge_type, weight=1.0, meta=""):
    if pd.isna(source) or pd.isna(target):
        return

    edges.append({
        "source": source,
        "target": target,
        "edge_type": edge_type,
        "weight": weight,
        "meta": meta
    })

# message -> cluster / domain / message_type / date
for _, row in message_df.iterrows():
    msg_node = make_node_id("message", row["message_id"])

    add_edge(
        msg_node,
        make_node_id("cluster", row["final_cluster"]),
        "belongs_to_cluster"
    )

    add_edge(
        msg_node,
        make_node_id("domain", row["source_domain"]),
        "from_source_domain"
    )

    add_edge(
        msg_node,
        make_node_id("message_type", row["message_type"]),
        "has_message_type"
    )

    if pd.notna(row["date"]):
        add_edge(
            msg_node,
            make_node_id("date", row["date"]),
            "occurred_on"
        )

# message -> ORG / Product / Part
for _, row in message_part_map.dropna(subset=["message_id"]).iterrows():
    msg_node = make_node_id("message", row["message_id"])

    if pd.notna(row.get("ORG")):
        add_edge(
            msg_node,
            make_node_id("org", row["ORG"]),
            "related_to_org"
        )

    if pd.notna(row.get("Product")):
        add_edge(
            msg_node,
            make_node_id("product", row["Product"]),
            "related_to_product"
        )

    if pd.notna(row.get("part_no")):
        add_edge(
            msg_node,
            make_node_id("part_no", row["part_no"]),
            "related_to_part_no"
        )

    if pd.notna(row.get("part_desc")):
        add_edge(
            msg_node,
            make_node_id("part_desc", row["part_desc"]),
            "related_to_part_desc"
        )

In [ ]:
temp = message_df.copy()
temp["_date_sort"] = pd.to_datetime(temp["date"], errors="coerce")
temp["_message_sort"] = temp["message_id"].astype(str)

temp = temp.sort_values(["source_domain", "_date_sort", "_message_sort"])

for domain, g in temp.groupby("source_domain"):
    ids = g["message_id"].tolist()

    for a, b in zip(ids[:-1], ids[1:]):
        add_edge(
            make_node_id("message", a),
            make_node_id("message", b),
            "next_message_in_domain",
            meta=f"domain={domain}"
        )

In [ ]:
cluster_transition = (
    temp
    .groupby("source_domain")
    .apply(lambda g: list(g["final_cluster"]))
)

transition_counts = {}

for domain, clusters in cluster_transition.items():
    for a, b in zip(clusters[:-1], clusters[1:]):
        key = (a, b, domain)
        transition_counts[key] = transition_counts.get(key, 0) + 1

for (a, b, domain), weight in transition_counts.items():
    add_edge(
        make_node_id("cluster", a),
        make_node_id("cluster", b),
        "cluster_transition_in_domain",
        weight=weight,
        meta=f"domain={domain}"
    )

In [ ]:
edges_df = (
    pd.DataFrame(edges)
    .drop_duplicates(subset=["source", "target", "edge_type", "meta"])
    .reset_index(drop=True)
)

print("edges_df:", edges_df.shape)
display(edges_df.head())

print("edge_type count")
display(edges_df["edge_type"].value_counts())

In [ ]:
node_schema = pd.DataFrame([
    ["message", "각 로그/메시지 행", "message_id, message_text"],
    ["cluster", "KMeans k=10 기반 의미 군집", "final_cluster"],
    ["source_domain", "HEAD, ITEM, DR, RULE 출처 도메인", "source_domain"],
    ["message_type", "메시지 유형", "message_type"],
    ["date", "발생일", "date"],
    ["org", "조직/법인/거점", "ORG"],
    ["product", "제품군", "Product"],
    ["part_no", "품목 번호", "PartNo, Part No."],
    ["part_desc", "품목 설명", "Description, Desc."]
], columns=["node_type", "description", "source_column"])

edge_schema = pd.DataFrame([
    ["belongs_to_cluster", "message", "cluster", "메시지가 특정 의미 군집에 속함"],
    ["from_source_domain", "message", "source_domain", "메시지 출처 도메인"],
    ["has_message_type", "message", "message_type", "메시지 유형"],
    ["occurred_on", "message", "date", "메시지 발생일"],
    ["related_to_org", "message", "org", "메시지가 특정 ORG와 연결됨"],
    ["related_to_product", "message", "product", "메시지가 특정 제품군과 연결됨"],
    ["related_to_part_no", "message", "part_no", "메시지가 특정 품목 번호와 연결됨"],
    ["related_to_part_desc", "message", "part_desc", "메시지가 특정 품목 설명과 연결됨"],
    ["next_message_in_domain", "message", "message", "동일 도메인 내 시간/순서상 다음 메시지"],
    ["cluster_transition_in_domain", "cluster", "cluster", "동일 도메인 내 군집 간 순차 전이"]
], columns=["edge_type", "source_node", "target_node", "meaning"])

display(node_schema)
display(edge_schema)

## 9. Export graph-ready artifacts

In [ ]:
cluster_profile_path = OUT_DIR / "cluster_profile_for_interpretation.xlsx"
nodes_path = OUT_DIR / "gnn_nodes.csv"
edges_path = OUT_DIR / "gnn_edges.csv"
node_schema_path = OUT_DIR / "gnn_node_schema.csv"
edge_schema_path = OUT_DIR / "gnn_edge_schema.csv"
part_long_path = OUT_DIR / "part_long_from_originals.csv"
message_part_map_path = OUT_DIR / "message_part_map.csv"
message_final_path = OUT_DIR / "message_with_cluster_name.csv"

with pd.ExcelWriter(cluster_profile_path, engine="openpyxl") as writer:
    cluster_profile.to_excel(writer, sheet_name="cluster_profile", index=False)
    cluster_keywords.to_excel(writer, sheet_name="cluster_keywords", index=False)
    cluster_examples.to_excel(writer, sheet_name="cluster_examples", index=False)
    node_schema.to_excel(writer, sheet_name="node_schema", index=False)
    edge_schema.to_excel(writer, sheet_name="edge_schema", index=False)

message_df.to_csv(message_final_path, index=False, encoding="utf-8-sig")
nodes_df.to_csv(nodes_path, index=False, encoding="utf-8-sig")
edges_df.to_csv(edges_path, index=False, encoding="utf-8-sig")
node_schema.to_csv(node_schema_path, index=False, encoding="utf-8-sig")
edge_schema.to_csv(edge_schema_path, index=False, encoding="utf-8-sig")
part_long.to_csv(part_long_path, index=False, encoding="utf-8-sig")
message_part_map.to_csv(message_part_map_path, index=False, encoding="utf-8-sig")

print("저장 완료")
print("cluster_profile:", cluster_profile_path)
print("message_final:", message_final_path)
print("nodes:", nodes_path)
print("edges:", edges_path)
print("part_long:", part_long_path)
print("message_part_map:", message_part_map_path)